# Met style counts under label map v0

CIS631 WikiArt-to-museum study. This notebook reads **metadata only** from the Met Collection API (it downloads no images). It labels each public-domain painting with a style under label map v0 and counts how many images each style gets.

Rules, applied in order:
1. **Artist anchor.** If the artist has at least 10 WikiArt images and at least 70% of them are in one shared class, the painting gets that class.
2. **Date and region fallback.** Used only for the pre-1780 classes and Ukiyo-e.
3. **Late copies excluded.** "Copy after", "Follower of" and similar works are excluded when they were made more than 50 years after the artist died.
4. **Everything else excluded.** The reason is recorded, so the exclusions can be counted.

Object JSON is cached in `met_out/cache/`, so if a run is interrupted, rerunning picks up where it stopped. Set `MAX_OBJECTS = 300` for a quick smoke test first.

In [ ]:
%pip install -q requests pandas

## Settings

In [ ]:
DEPARTMENTS = [11, 1, 21]   # 11 European Paintings, 1 American Wing, 21 Modern and Contemporary Art
UKIYOE = True               # also search Asian Art for Japanese woodblock prints
MAX_OBJECTS = None          # e.g. 300 for a smoke test
RATE = 40.0                 # requests per second (Met asks for <= 80)
WORKERS = 8
OUT = "met_out"
PAINTINGS_ONLY = True       # pre-filter with the Met's bulk CSV so only paintings (and Japanese prints) are fetched
METOBJECTS_CSV_URL = "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv"

## Imports and label map v0

In [ ]:
import csv
import json
import re
import sys
import threading
import time
import unicodedata
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests

API = "https://collectionapi.metmuseum.org/public/collection/v1"
SEARCH_V11 = "https://collectionapi.metmuseum.org/public/collection/v1.1/search"
WIKIART_CSVS = [
    "https://raw.githubusercontent.com/cs-chan/ArtGAN/master/WikiArt%20Dataset/Style/style_train.csv",
    "https://raw.githubusercontent.com/cs-chan/ArtGAN/master/WikiArt%20Dataset/Style/style_val.csv",
]

# Departments: 11 European Paintings, 1 The American Wing, 21 Modern and Contemporary Art
DEFAULT_DEPARTMENTS = [11, 1, 21]
ASIAN_ART = 6

MIN_TEST_IMAGES = 30
TEST_FRACTION = 0.70
ANCHOR_MIN_IMAGES = 10
ANCHOR_MIN_SHARE = 0.70

# Label map v0: WikiArt style -> shared class. Styles not listed are dropped.
LABEL_MAP_V0 = {
    "Early_Renaissance": "Early Renaissance",
    "High_Renaissance": "High Renaissance and Mannerism",
    "Mannerism_Late_Renaissance": "High Renaissance and Mannerism",
    "Northern_Renaissance": "Northern Renaissance",
    "Baroque": "Baroque",
    "Rococo": "Rococo",
    "Romanticism": "Romanticism",
    "Realism": "Realism",
    "Impressionism": "Impressionism",
    "Post_Impressionism": "Post-Impressionism",
    "Pointillism": "Post-Impressionism",
    "Symbolism": "Symbolism",
    "Ukiyo_e": "Ukiyo-e",
}
SHARED_CLASSES = sorted(set(LABEL_MAP_V0.values()))

ITALIAN = {"italian", "florentine", "venetian", "sienese", "neapolitan", "bolognese",
           "roman", "lombard", "ferrarese", "umbrian", "milanese", "genoese",
           "veronese", "paduan", "emilian", "marchigian", "piedmontese"}
NORTHERN = {"netherlandish", "flemish", "dutch", "german", "french", "austrian",
            "swiss", "bohemian", "south netherlandish", "north netherlandish"}
EUROPEAN = ITALIAN | NORTHERN | {"spanish", "british", "english", "scottish", "irish",
                                 "portuguese", "danish", "swedish", "norwegian",
                                 "polish", "russian", "hungarian", "czech", "belgian"}
COPY_PREFIXES = ("copy after", "follower of", "imitator of", "style of", "after")

## API and WikiArt helpers

In [ ]:
class RateLimiter:
    def __init__(self, per_second):
        self.interval = 1.0 / per_second
        self.lock = threading.Lock()
        self.next_time = time.monotonic()

    def wait(self):
        with self.lock:
            now = time.monotonic()
            if self.next_time > now:
                time.sleep(self.next_time - now)
            self.next_time = max(now, self.next_time) + self.interval


def slugify(name):
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def artist_slugs(display_name):
    """Candidate WikiArt slugs for a Met artistDisplayName.

    'Rembrandt (Rembrandt van Rijn)' -> ['rembrandt', 'rembrandt-van-rijn']
    """
    name = display_name.split("|")[0].strip()
    if not name:
        return []
    candidates = [re.sub(r"\(.*?\)", "", name).strip()]
    candidates += re.findall(r"\((.*?)\)", name)
    slugs = []
    for c in candidates:
        s = slugify(c)
        if s and s not in slugs:
            slugs.append(s)
        stripped = re.sub(r"-the-(elder|younger)$", "", s)
        if stripped and stripped not in slugs:
            slugs.append(stripped)
    return slugs


def load_wikiart(cache_dir):
    """Return {artist_slug: Counter(shared class or 'DROPPED')} and {artist_slug: set(title slugs)}."""
    styles = defaultdict(Counter)
    titles = defaultdict(set)
    for url in WIKIART_CSVS:
        path = cache_dir / Path(url).name
        if not path.exists():
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            path.write_bytes(r.content)
        with path.open() as f:
            for row in csv.reader(f):
                style, filename = row[0].split("/", 1)
                artist, _, title = filename.rsplit(".", 1)[0].partition("_")
                styles[artist][LABEL_MAP_V0.get(style, "DROPPED")] += 1
                titles[artist].add(re.sub(r"-?\(?\d+\)?$", "", title))
    return styles, titles


def get_json(session, limiter, url, params=None, retries=5):
    for attempt in range(retries):
        limiter.wait()
        try:
            r = session.get(url, params=params, timeout=30)
            if r.status_code == 404:
                return None
            if r.status_code in (403, 429, 500, 502, 503):
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            time.sleep(2 ** attempt)
    raise RuntimeError(f"giving up on {url} {params or ''}")


def department_object_ids(session, limiter, dept):
    data = get_json(session, limiter, f"{API}/objects", {"departmentIds": dept})
    return (data or {}).get("objectIDs") or []


def ukiyoe_candidate_ids(session, limiter):
    """Best effort: search Asian Art for woodblock prints with images."""
    params = {"departmentId": ASIAN_ART, "hasImages": "true", "q": "woodblock print"}
    ids, offset, limit = [], 0, 500
    while True:
        data = get_json(session, limiter, SEARCH_V11, {**params, "offset": offset, "limit": limit})
        batch = (data or {}).get("objectIDs") or []
        new = [i for i in batch if i not in set(ids)]
        if not new:  # empty page, or the API ignored offset and repeated a page
            break
        ids += new
        if len(ids) >= (data or {}).get("total", float("inf")):
            break
        offset += len(batch)
    if not ids:  # v1.1 unavailable or shaped differently: try the old endpoint
        data = get_json(session, limiter, f"{API}/search", params)
        ids = (data or {}).get("objectIDs") or []
    return ids


def fetch_object(session, limiter, cache_dir, object_id):
    path = cache_dir / f"{object_id}.json"
    if path.exists():
        return json.loads(path.read_text())
    obj = get_json(session, limiter, f"{API}/objects/{object_id}") or {}
    path.write_text(json.dumps(obj))
    return obj

## Labeling rules

In [ ]:
def year(value):
    m = re.search(r"-?\d{3,4}", str(value or ""))
    return int(m.group()) if m else None


def region(obj):
    nat = (obj.get("artistNationality") or "").split("|")[0].strip().lower()
    nat = re.sub(r",.*$", "", nat)
    if nat in ITALIAN:
        return "italian"
    if nat in NORTHERN:
        return "northern"
    if nat in EUROPEAN or obj.get("department") == "European Paintings":
        return "european"
    if "japan" in (obj.get("culture") or "").lower():
        return "japanese"
    return None


def date_fallback(obj, mid, reg):
    is_print = obj.get("classification", "").lower().startswith("print")
    if reg == "japanese" and is_print and 1680 <= mid <= 1900:
        return "Ukiyo-e"
    if is_print:
        return None
    if reg == "italian" and 1400 <= mid < 1490:
        return "Early Renaissance"
    if reg == "italian" and 1490 <= mid < 1600:
        return "High Renaissance and Mannerism"
    if reg == "northern" and 1400 <= mid < 1580:
        return "Northern Renaissance"
    if reg in ("italian", "northern", "european") and 1600 <= mid < 1720:
        return "Baroque"
    if reg in ("italian", "northern", "european") and 1720 <= mid < 1780:
        return "Rococo"
    return None


def label(obj, wikiart_styles, wikiart_titles):
    """Return (label or None, rule or exclusion reason, matched slug, title match)."""
    begin, end = obj.get("objectBeginDate"), obj.get("objectEndDate")
    mid = (begin + end) / 2 if isinstance(begin, int) and isinstance(end, int) else None

    prefix = (obj.get("artistPrefix") or "").strip().lower()
    death = year(obj.get("artistEndDate"))
    if prefix.startswith(COPY_PREFIXES) and death and begin and begin > death + 50:
        return None, "excluded: late copy or follower", "", False

    slug = next((s for s in artist_slugs(obj.get("artistDisplayName") or "")
                 if s in wikiart_styles), "")
    title_match = bool(slug) and slugify(obj.get("title") or "") in wikiart_titles.get(slug, ())

    if slug:
        counts = wikiart_styles[slug]
        total = sum(counts.values())
        top, n = counts.most_common(1)[0]
        if total >= ANCHOR_MIN_IMAGES and top != "DROPPED" and n / total >= ANCHOR_MIN_SHARE:
            return top, "artist anchor", slug, title_match

    if mid is None:
        return None, "excluded: no date", slug, title_match
    reg = region(obj)
    fallback = date_fallback(obj, mid, reg)
    if fallback:
        return fallback, "date and region", slug, title_match
    if mid >= 1780:
        reason = "excluded: post-1780, artist mixed-style" if slug else "excluded: post-1780, artist not in WikiArt"
    elif reg is None:
        reason = "excluded: unknown region"
    else:
        reason = "excluded: outside fallback windows"
    return None, reason, slug, title_match


def eligible(obj, include_prints):
    if not obj.get("isPublicDomain") or not obj.get("primaryImage"):
        return False
    cls = (obj.get("classification") or "").lower()
    if cls == "paintings":
        return True
    return include_prints and cls.startswith("print") and "japan" in (obj.get("culture") or "").lower()

## Load WikiArt artist styles

In [ ]:
out = Path(OUT)
cache_dir = out / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
session = requests.Session()
session.headers["User-Agent"] = "CIS631-wikiart-museum-study/0.1 (metadata count, research)"
limiter = RateLimiter(RATE)

wikiart_styles, wikiart_titles = load_wikiart(out)
print(f"{len(wikiart_styles)} WikiArt artists")

## Fetch Met object metadata

## Pre-filter to paintings (optional)

Skips the department and search listings' non-paintings. If the CSV download fails, the next cell fetches everything as before.

In [ ]:
# Pre-filter: the Met's bulk CSV (about 300 MB, one download) lists every object's
# department, classification, culture and public-domain flag, so we fetch only
# public-domain paintings from DEPARTMENTS, plus Japanese prints from Asian Art.
candidates = None
if PAINTINGS_ONLY:
    import pandas as pd
    csv_path = out / "MetObjects.csv"
    try:
        if not csv_path.exists():
            with requests.get(METOBJECTS_CSV_URL, stream=True, timeout=120) as r:
                r.raise_for_status()
                with csv_path.open("wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        meta = pd.read_csv(csv_path, low_memory=False,
                           usecols=["Object ID", "Is Public Domain", "Department", "Classification", "Culture"])
        dept_names = {d: n for d, n in {11: "European Paintings", 1: "The American Wing", 21: "Modern and Contemporary Art"}.items() if d in DEPARTMENTS}
        pd_mask = meta["Is Public Domain"].astype(str).str.lower() == "true"
        cls = meta["Classification"].fillna("").str.lower()
        paintings = pd_mask & (cls == "paintings") & meta["Department"].isin(dept_names.values())
        prints = (pd_mask & cls.str.startswith("print") & (meta["Department"] == "Asian Art")
                  & meta["Culture"].fillna("").str.contains("Japan", case=False))
        candidates = set(meta.loc[paintings, "Object ID"].astype(int))
        print(f"Public-domain paintings in {list(dept_names.values())}: {len(candidates)}")
        if UKIYOE:
            jp = set(meta.loc[prints, "Object ID"].astype(int))
            print(f"Public-domain Japanese prints in Asian Art: {len(jp)}")
            candidates |= jp
    except Exception as e:
        print(f"Pre-filter failed ({e}); falling back to fetching every object in the departments.")
        candidates = None

In [ ]:
ids = []
for dept in (DEPARTMENTS if candidates is None else []):
    dept_ids = department_object_ids(session, limiter, dept)
    print(f"Department {dept}: {len(dept_ids)} objects")
    ids += dept_ids
if UKIYOE and candidates is None:
    uk = ukiyoe_candidate_ids(session, limiter)
    print(f"Ukiyo-e search: {len(uk)} candidates")
    ids += uk
ids = list(dict.fromkeys(ids))
if candidates is not None:
    ids = [i for i in ids if i in candidates] + sorted(candidates - set(ids))
    print(f"After pre-filter: {len(ids)} objects to fetch")
ids = ids[:MAX_OBJECTS]

objects = []
with ThreadPoolExecutor(WORKERS) as pool:
    futures = [pool.submit(fetch_object, session, limiter, cache_dir, i) for i in ids]
    for n, fut in enumerate(as_completed(futures), 1):
        objects.append(fut.result())
        if n % 500 == 0 or n == len(futures):
            print(f"fetched {n}/{len(futures)}")

## Label and save

In [ ]:
import pandas as pd

rows, reasons = [], Counter()
for obj in objects:
    if not eligible(obj, UKIYOE):
        continue
    lab, rule, slug, title_match = label(obj, wikiart_styles, wikiart_titles)
    reasons[rule] += 1
    rows.append({
        "objectID": obj["objectID"],
        "department": obj.get("department", ""),
        "classification": obj.get("classification", ""),
        "title": obj.get("title", ""),
        "artistPrefix": obj.get("artistPrefix", ""),
        "artistDisplayName": obj.get("artistDisplayName", ""),
        "artistNationality": obj.get("artistNationality", ""),
        "objectBeginDate": obj.get("objectBeginDate", ""),
        "objectEndDate": obj.get("objectEndDate", ""),
        "label_v0": lab or "",
        "rule": rule,
        "wikiart_artist": slug,
        "possible_wikiart_duplicate": title_match,
        "primaryImage": obj.get("primaryImage", ""),
        "objectURL": obj.get("objectURL", ""),
    })

df = pd.DataFrame(rows).sort_values("objectID")
df.to_csv(out / "met_objects.csv", index=False)
labeled = df[df.label_v0 != ""]
print(f"Eligible public-domain objects with images: {len(df)}")
print(f"Labeled under v0: {len(labeled)}  (excluded: {len(df) - len(labeled)})")

## Per-class counts and floors

In [ ]:
summary = (labeled.groupby("label_v0")
           .agg(n_labeled=("objectID", "size"),
                n_artists=("artistDisplayName", "nunique"),
                possible_wikiart_duplicates=("possible_wikiart_duplicate", "sum"))
           .reindex(SHARED_CLASSES, fill_value=0))
summary["est_test_70pct"] = (summary.n_labeled * TEST_FRACTION).astype(int)
summary["est_finetune_30pct"] = summary.n_labeled - summary.est_test_70pct
summary["evaluable"] = summary.est_test_70pct >= MIN_TEST_IMAGES
summary.index.name = "class"
summary.to_csv(out / "met_class_counts.csv")

if len(labeled):
    top = summary.n_labeled.idxmax()
    k = int(summary.evaluable.sum())
    print(f"Majority floor: {top} = {summary.n_labeled.max() / len(labeled):.1%} of labeled objects")
    if k:
        print(f"Chance floor over {k} evaluable classes: {1 / k:.1%}")
summary

## Rules and exclusion reasons

In [ ]:
pd.DataFrame(reasons.most_common(), columns=["rule_or_reason", "n"]).to_csv(out / "met_rule_counts.csv", index=False)
pd.DataFrame(reasons.most_common(), columns=["rule_or_reason", "n"])

## Mixed-style artists

These artists split between WikiArt styles below the 70% bar, so their works are excluded. They are candidates for hand labeling by period.

In [ ]:
mixed = df[df.rule == "excluded: post-1780, artist mixed-style"].wikiart_artist.value_counts().head(15)
pd.DataFrame({"objects": mixed,
              "wikiart_styles": [dict(wikiart_styles[s].most_common(3)) for s in mixed.index]})